# Phase 4 — 优化方案选择器

**目标**：勾选 / 预设 MixUp 模块，生成 `run_id`、配置快照与报告草稿。  
**默认**：B + CPPO 开启；加开其它模块时二者保持，除非显式 `False`。  
**开训**：本 Notebook **不强制**跑 DeepSpeed；需要时再复用 Phase1 `run_training`。

预设别名：`m1` / `speed` / `m5` / `m5q` / `vanilla` / `b_only`（见 `mixup.presets.PRESET_ALIASES`）。

## 0. 环境（Colab）

In [ ]:
import sys
from pathlib import Path

# Colab: 先挂载 Drive，再设 PROJECT_DIR
PROJECT_DIR = Path("/content/drive/MyDrive/MixUpLLaVA-video-r1")
MIXUP_REPO = PROJECT_DIR / "repo" / "MixUpLLaVA-Video-R1"
# 本地开发可改为仓库根目录：
if not MIXUP_REPO.is_dir():
    MIXUP_REPO = Path.cwd()
    if not (MIXUP_REPO / "mixup").is_dir():
        MIXUP_REPO = Path.cwd().parent  # notebooks/ 下打开时

sys.path.insert(0, str(MIXUP_REPO))
print("MIXUP_REPO =", MIXUP_REPO)

## 1. 选择策略

- 方式 A：`PRESET = "m1"`（默认 B+CPPO）
- 方式 B：在 `MIXUP` 里叠开关，例如 `{"ngrpo": True}`
- 方式 C：消融显式关掉默认，例如 `{"project_baseline": False}`（仅 CPPO）

In [ ]:
from mixup.presets import PRESET_ALIASES, list_presets
from mixup.training_entry import prepare_training, describe_launch

print("aliases:", PRESET_ALIASES)
print("files:", list_presets())

PRESET = "m1"          # m1 | speed | m5 | m5q | vanilla | b_only
MIXUP = {
    # "ngrpo": True,    # 叠在 B+CPPO 上
    # "gfpo": True, "gfpo_top_k": 4,
    # "mo_grpo": True,
    # "project_baseline": False,  # 显式去 B
    # "cppo": False,              # 显式去 CPPO
}
TAG = ""               # 可选后缀
APPLY_PATCH = False    # Colab 且确认 REPO 路径后再 True
DRY_RUN = False        # True=只生成 plan、不写盘

plan = prepare_training(
    preset=PRESET,
    mixup=MIXUP or None,
    project_dir=PROJECT_DIR if PROJECT_DIR.is_dir() else None,
    mixup_repo=MIXUP_REPO,
    apply_patch=APPLY_PATCH,
    dry_run=DRY_RUN,
    tag=TAG,
    notes="Phase4 selector",
    namespace=globals(),
)
print(plan.summary())
print(describe_launch(plan))

## 2. （可选）覆盖训练档位

默认加载 `configs/colab_c1.yaml`。改步数等用 `tier_overrides`：

In [ ]:
# 示例：冒烟 10 step（不与 Phase1 50-step 混比）
# plan = prepare_training(
#     preset="m1",
#     mixup={"ngrpo": True},
#     project_dir=PROJECT_DIR if PROJECT_DIR.is_dir() else None,
#     mixup_repo=MIXUP_REPO,
#     tier_overrides={"max_steps": 10},
#     tag="smoke10",
#     namespace=globals(),
# )
# print(plan.summary())

## 3. 开训（延后 / 按需）

将 Phase1 Notebook 的 `run_training` 指向 `RUN_OUTPUT_DIR`，并确认已 `APPLY_PATCH=True`。  
训完后填写 `plan.report_stub_path` 或复制 `docs/PHASE4_REPORT.md`。

In [ ]:
print("RUN_ID =", globals().get("RUN_ID"))
print("OUT    =", globals().get("RUN_OUTPUT_DIR"))
print("enabled=", plan.enabled)
print("dropped=", plan.dropped_project_defaults)
# from phase1 notebook: run_training(str(RUN_OUTPUT_DIR), plan.run_id)